In [1]:
import watermark
pkg_versions = watermark.watermark(
    packages="requests,pandas,tqdm,geopandas")
print(pkg_versions)

requests : 2.32.5
pandas   : 1.5.3
tqdm     : 4.67.1
geopandas: 1.0.1



In [2]:
import os
import requests
from dotenv import load_dotenv
import pandas as pd
from tqdm import tqdm
from io import StringIO
import xml.etree.ElementTree as ET
import geopandas as gpd

load_dotenv()
API_KEY = os.getenv("API_KEY")
BASE_URL = "https://apis.data.go.kr"

# 행정안전부_행정표준코드_법정동코드

In [3]:
URL = f"{BASE_URL}/1741000/StanReginCd/getStanReginCdList"
params = {
    "serviceKey":API_KEY,
    "numOfRows": 1000,
    "pageNo": 1,
    "flag":"Y",
    "locatadd_nm":"부산광역시",
    "type":"json"
}
res = requests.get(URL, params= params)
data = res.json()
station_df = pd.DataFrame(data["StanReginCd"][1]["row"])
station_df["signguCode"] = (
    station_df["sido_cd"].astype(str).str.zfill(2)
    + station_df["sgg_cd"].astype(str).str.zfill(3)
)
busan_df = station_df[
    station_df["locallow_nm"].str.contains(
        r".*(?:구|군)$", na=False
    )
].reset_index(drop=True)

In [4]:
busan_df

,region_cd,sido_cd,sgg_cd,umd_cd,ri_cd,locatjumin_cd,locatjijuk_cd,locatadd_nm,locat_order,locat_rm,locathigh_cd,locallow_nm,adpt_de,signguCode
0,2611000000,26,110,000,00,2611000000,2611000000,부산광역시 중구,1,,2600000000,중구,,26110
1,2614000000,26,140,000,00,2614000000,2614000000,부산광역시 서구,2,,2600000000,서구,,26140
2,2617000000,26,170,000,00,2617000000,2617000000,부산광역시 동구,3,,2600000000,동구,,26170
3,2620000000,26,200,000,00,2620000000,2620000000,부산광역시 영도구,4,,2600000000,영도구,,26200
4,2623000000,26,230,000,00,2623000000,2623000000,부산광역시 부산진구,5,,2600000000,부산진구,,26230
5,2626000000,26,260,000,00,2626000000,2626000000,부산광역시 동래구,6,,2600000000,동래구,,26260
6,2629000000,26,290,000,00,2629000000,2629000000,부산광역시 남구,7,,2600000000,남구,,26290
7,2632000000,26,320,000,00,2632000000,2632000000,부산광역시 북구,8,,2600000000,북구,,26320
8,2635000000,26,350,000,00,2635000000,2635000000,부산광역시 해운대구,9,,2600000000,해운대구,,26350
9,2638000000,26,380,000,00,2638000000,2638000000,부산광역시 사하구,10,,2600000000,사하구,,26380


In [5]:
busan_codes = busan_df["signguCode"].unique()

# 한국관광공사_방한 외래관광객 국적별 성별 집계

In [6]:
BASE_URL = "https://api.odcloud.kr/api/15136295/v1/uddi:8c7e5df9-d3d3-4d3f-83d3-f360dd49a143"
params = {
    "page": 1,
    "perPage": 300,
    "serviceKey": API_KEY
}

headers = {
    "accept": "*/*",
    "Authorization": "*/*"
}

res = requests.get(
    BASE_URL,
    params=params,
    headers=headers
)
data = res.json()
df = pd.DataFrame(data["data"])

In [7]:
df.to_csv("외래관광객_국적별_성별_집계.csv", index=False)

# 한국관광공사_지역별 관광 다양성(부적합)
- 지역별 국제적 다양성 정보 목록 조회
- 외국인 소비액, 외국인 국적별 방문자 수, 외국인 방문객 국적 다양성 세부 지표

In [23]:
BASE_URL = "https://apis.data.go.kr"
URL = f"{BASE_URL}/B551011/AreaTarDivService/areaIntlDivList"

In [24]:
params = {
    "serviceKey":API_KEY,
    "MobileApp":"AppTest",
    "MobileOS":"ETC",
    "pageNo": 1,
    "numOfRows": 10,
    "baseYm":"202504",
    "areaCd":"26",
    "intlDivIxCd":"3301",
    # 3301: 외국인 소비액, 3302: 방문자수, 3303: 국적 다양성
    "_type":"json",    
}

In [25]:
res = requests.get(URL, params= params)
data = res.json()
total_count = data["response"]["body"]["totalCount"]

In [11]:
st_date = pd.to_datetime("2010-01-01")
ed_date = pd.to_datetime("2026-09-01")

In [12]:
num_of_rows = 100
params.update({"numOfRows":num_of_rows})
ym_list = pd.date_range(st_date, ed_date, freq="MS")
dfs = list()
for date in tqdm(ym_list):
    try:
        params.update({"baseYm":date.strftime("%Y%m")})
        res = requests.get(URL, params= params)
        data = res.json()
        df = pd.DataFrame(
            data["response"]["body"]["items"]["item"])
        dfs.append(df)
    except:
        print(date)

  1%|          | 2/201 [00:00<00:27,  7.23it/s]

2010-01-01 00:00:00
2010-02-01 00:00:00


  2%|▏         | 4/201 [00:00<00:28,  7.03it/s]

2010-03-01 00:00:00
2010-04-01 00:00:00


  3%|▎         | 6/201 [00:00<00:29,  6.53it/s]

...

2019-09-01 00:00:00
2019-10-01 00:00:00


 60%|█████▉    | 120/201 [00:16<00:11,  6.88it/s]

2019-11-01 00:00:00
2019-12-01 00:00:00


100%|██████████| 201/201 [00:28<00:00,  7.16it/s]

2026-08-01 00:00:00
2026-09-01 00:00:00


In [14]:
df = pd.concat(dfs,ignore_index=True)
df.head()

,baseYm,areaCd,areaNm,signguCd,signguNm,intlDivIxCd,intlDivIxNm,intlDivIxVal
0,202001,26,부산광역시,0,_,3301,외국인 소비액,75.82
1,202001,26,부산광역시,26110,중구,3301,외국인 소비액,75.92
2,202001,26,부산광역시,26140,서구,3301,외국인 소비액,70.91
3,202001,26,부산광역시,26170,동구,3301,외국인 소비액,70.41
4,202001,26,부산광역시,26200,영도구,3301,외국인 소비액,70.23


In [15]:
df.to_csv("월별 시군구별 외국인 소비액.csv", index=False)

In [16]:
params.update({"intlDivIxCd":"3302"})

In [17]:
num_of_rows = 100
params.update({"numOfRows":num_of_rows})
ym_list = pd.date_range(st_date, ed_date, freq="MS")
dfs = list()
for date in tqdm(ym_list):
    try:
        params.update({"baseYm":date.strftime("%Y%m")})
        res = requests.get(URL, params= params)
        data = res.json()
        df = pd.DataFrame(
            data["response"]["body"]["items"]["item"])
        dfs.append(df)
    except:
        pass
df = pd.concat(dfs,ignore_index=True)
df.head()

100%|██████████| 201/201 [00:28<00:00,  7.03it/s]


,baseYm,areaCd,areaNm,signguCd,signguNm,intlDivIxCd,intlDivIxNm,intlDivIxVal
0,202001,26,부산광역시,0,_,3302,외국인 방문자수,77.3
1,202001,26,부산광역시,26110,중구,3302,외국인 방문자수,74.78
2,202001,26,부산광역시,26140,서구,3302,외국인 방문자수,75.48
3,202001,26,부산광역시,26170,동구,3302,외국인 방문자수,73.13
4,202001,26,부산광역시,26200,영도구,3302,외국인 방문자수,74.5


In [19]:
df.to_csv("월별 시군구별 외국인 국적별 방문자수.csv", index=False)